In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import glob
import os
import pandas as pd
import multiprocessing
import logging
import blimpy as bl 
%matplotlib inline

In [2]:
df = pd.read_csv('/datax/scratch/benjb/bl_nearby_stars/BL_cadences_unique_nearby_star_sample_only.csv')
df.insert(0, column='Index', value=np.arange(len(df)))

spliced_nums = []
for i, h5 in enumerate(df['.h5 path 1']):
    if 'spliced' in h5:
        spliced_nums.append(i)

unspliced_nums = []
for i, h5 in enumerate(df['.h5 path 1']):
    if not 'spliced' in h5:
        unspliced_nums.append(i)

# skip_idx = [0, 1, 2, 63, 64, 65, 95, 96, 97, 121, 122, 123, 147, 148, 149,
#              156, 157, 171, 172, 9178, 9179, 9180, 9197, 9198, 9199, 9200, 9201, 9202, 9203, 9204,
#              9205, 9206, 9207, 9208, 9209, 9210, 9241, 9242, 19474, 19475, 19499, 19571, 19578, 19579, 
#              19580, 19581, 19582, 19583, 19584, 19585, 28806, 28807, 28831, 28832, 28856, 28857, 28858, 
#              28882, 28883, 28884, 28885, 28886, 28887, 28917, 28918, 28919, 28920, 28921, 28922, 28923]

# skip_nums = np.concatenate([unspliced_nums, skip_idx])
# skip_nums = np.sort(skip_nums)

# print(len(df))

# df.drop(index=skip_nums, inplace=True)

# print(len(df))

# check_idx = np.load('/datax/scratch/benjb/bl_nearby_stars/blpc1_spliced_files_idx_100525.npz')['arr_0']
# # check_idx = check_idx[check_idx > 26947]
# check_idx = [6706, 9493, 13168, 13169]
# print(check_idx)
# print(len(check_idx))

# # skip_nums = np.concatenate([unspliced_nums, skip_idx])
# # skip_nums = np.sort(skip_nums)

# print(len(df))

# # NEED TO RUN THIS SEARCH ON ALL FILES IN SPLICED DIRECTORY -- FIGURE OUT WHICH IDX THESE ARE
# df = df.iloc[check_idx]
# #df.drop(index=skip_nums, inplace=True)

# print(len(df))

check_idx = np.load('/mnt/blpc1/datax/scratch/benjb/bl_nearby_stars/blpc1_spliced_files_idx_100525.npz')['arr_0']
#check_idx = check_idx[check_idx > 26947]
print(check_idx)
print(len(check_idx))

# skip_nums = np.concatenate([unspliced_nums, skip_idx])
# skip_nums = np.sort(skip_nums)

print(len(df))

# NEED TO RUN THIS SEARCH ON ALL FILES IN SPLICED DIRECTORY -- FIGURE OUT WHICH IDX THESE ARE
df = df.iloc[check_idx]
#df.drop(index=skip_nums, inplace=True)

print(len(df))

[ 6706  9493 13168 ... 25426 25427 25428]
1396
39177
1396


In [ ]:
### FOR BLISS:

outdir = '/datax/scratch/benjb/bl_nearby_stars/blpc1_spliced/'
logdir = '/datax/scratch/benjb/bl_nearby_stars/bliss_logs/'

logger = logging.getLogger(__name__)

def process_files(xxx):
    n = xxx[0]
    dfbatch = xxx[1]
    snr = 20
    #pfb = '/datax/scratch/benjb/bl_nearby_stars/GBT_spliced_PFB_response.f32'
    # Remove all handlers associated with the root logger object.
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)
    logging.basicConfig(filename=f'{logdir}0_2_1_{n}_blpc1_062426.log', filemode="a", level=logging.DEBUG)
    # old_log = f'{outdir}000_{n}_060525.log'
    # with open(old_log) as f:
    #     f = f.readlines()
    # important = []
    # for line in f:
    #     if 'cadences' in line:
    #         important.append(line)
    # last_line = important[-1]
    # skip_number = int((last_line.split('#')[1]).split(' of ')[0])
    for i in range(len(dfbatch)):
        index = dfbatch.iloc[i]['Index']
        logger.info(f'Searching #{i+1} of {len(dfbatch)} cadences (index {index} of 39177 overall) ...')
        # if i+1 < skip_number:
        #     logger.info('Already searched this cadence. Continuing ...')
        #     continue
        # if i+1 in skip_nums:
        #     logger.info('Already searched this cadence. Continuing ...')
        #     continue
        test_file = dfbatch[f'.h5 path 1'].values[i]
        fb = bl.Waterfall(test_file, load_data=False)
        nfc = fb.header['nchans']
        # check for configuration
        if nfc % (2**20) == 0:
            # configuration is normal
            ncc = nfc // 2**20
            nfpc = 2**20
            config = 'u' # usual
            pfb = '/datax/scratch/benjb/bl_nearby_stars/GBT_spliced_PFB_response.f32'
        elif nfc % (1033216) == 0:
            ncc = nfc // 1033216
            nfpc = 1033216
            config = 'o' # old
            pfb = '/datax/scratch/benjb/bl_nearby_stars/GBT_spliced_PFB_response_1033216.f32'
        else:
            print('Unusual configuration; skipping for now.')
            continue
        for j in range(6): # for each h5 file in a cadence
            h5idx = j+1
            file = dfbatch[f'.h5 path {h5idx}'].values[i]
            logger.info(f'  Searching #{h5idx} of 6 files in this cadence ({file}) ...')
            if not 'spliced' in file:
                logger.info('  Unspliced file! Continuing ...')
                continue
            # run BLISS
            # try:
            #     console = f'bliss_find_hits {file} -e {pfb} -d cuda:{n} -md -4 -MD 4 -s {snr} --number-coarse 64 --distance 30 --output ' + outdir + f'{index}_{h5idx}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
            #     os.system(console)
            # run BLISS for each coarse channel
            for k in range(ncc):
                # check whether dat already exists:
                datcheck = outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
                if os.path.exists(datcheck):
                    with open(datcheck) as f:
                        lines = f.readlines()
                    if len(lines) > 0:
                        if k % 200 == 0:
                            logger.info(f'  Channel {k} of {ncc}: Already searched!')
                    else:
                        logger.info(f'  Channel {k} of {ncc}: Empty .dat. Re-searching ...')
                        try: # for spliced files, do one cc at a time
                            if k%200 == 0:
                                logger.info(f'  Channel {k} of {ncc}: Searching ...')
                            console = f'bliss_find_hits {file} -e {pfb} -d cuda:{n+2} -md -4 -MD 4 -s {snr} --number-coarse 1 -c {k} --nchan-per-coarse {nfpc} --distance 30 --output ' + outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
                            os.system(console)
                        except:
                            print('Search failed; check later.')
                if not os.path.exists(datcheck):
                    try: # for spliced files, do one cc at a time
                        if k%200 == 0:
                            logger.info(f'  Channel {k} of {ncc}: Searching ...')
                        console = f'bliss_find_hits {file} -e {pfb} -d cuda:{n+2} -md -4 -MD 4 -s {snr} --number-coarse 1 -c {k} --nchan-per-coarse {nfpc} --distance 30 --output ' + outdir + f'{index}_{h5idx}_{k}_' + os.path.basename(file)[:-3] + f'_nosig_nosk_SNR_{snr}_L1_30.dat'
                        os.system(console)
                    except:
                        print('Search failed; check later.')
                else:
                    if k % 200 == 0:
                        logger.info(f'  Channel {k} of {ncc}: Already searched!')

# if __name__ == "__main__":
#     # Define n sets of files
#     nnodes = 4 # number of GPUs on this node (misleadingly named)
#     nblpc = 3 # number of blpc nodes contributing to the search
#     batch_size = len(df) // nblpc // nnodes
#     file_sets = [[str(i), df.iloc[i*batch_size+len(df)*1//nblpc:(i+1)*batch_size+len(df)*1//nblpc]] for i in range(nnodes)]
#     dfimp = file_sets[3][1] # important df
#     print(len(dfimp))
#     dfcut = dfimp[233:]
#     batch_size = len(dfcut) // nblpc // nnodes
#     file_sets = [[str(i), dfcut.iloc[i*batch_size+len(dfcut)*1//nblpc:(i+1)*batch_size+len(dfcut)*1//nblpc]] for i in range(nnodes)]
#     #file_sets.append([str(nnodes), df.iloc[nnodes*batch_size+len(df)*0//4:len(df)*1//4]])

#     # Create a pool of n processes
#     with multiprocessing.Pool(processes=nnodes) as pool:
#         # Map the process_files function to each file set
#         pool.map(process_files, file_sets)

#     print("All files processed.")

if __name__ == "__main__":
    # Define n sets of files
    nnodes = 2
    # batch_size = len(df) // 3 // nnodes
    # file_sets = [[str(i), df.iloc[i*batch_size+len(df)*2//3:(i+1)*batch_size+len(df)*2//3]] for i in range(nnodes)] # run on all 4 GPUs

    batch_size = len(df) // nnodes 
    file_sets = [[str(i), df.iloc[i*batch_size:(i+1)*batch_size]] for i in range(nnodes)]

    #file_sets = [[str(i), df.iloc[i*batch_size+len(df)*2//3:(i+1)*batch_size+len(df)*2//3]] for i in range(nnodes-1)] # leave a GPU open
    #file_sets = [[str(i), df.iloc[i*batch_size:(i+1)*batch_size]] for i in range(nnodes)]
    #file_sets.append([str(nnodes), df.iloc[nnodes*batch_size+len(df)*3//4:len(df)]])

    # Create a pool of n processes
    with multiprocessing.Pool(processes=nnodes) as pool:
        # Map the process_files function to each file set
        pool.map(process_files, file_sets)

    print("All files processed.")